In [ ]:
import pandas as pd
import neuralset as ns
from neuralset.features import HuggingFaceText
import numpy as np
import string
import torch

df = pd.read_csv("datasets/svo_word_level.csv")

In [ ]:
words = df[["word", "sentence_id"]]
words

In [ ]:
def make_sentence(words):
    s = ""
    cum_s = []
    indices = []
    for word in words:
        if word in string.punctuation:
            s = s.strip()
        indices.append(len(s))
        s += word
        cum_s.append(s)
        s += " "

    return (s.strip(), indices, cum_s)

In [ ]:
def events_from_words(words: pd.DataFrame) -> pd.DataFrame:
    events = words.copy()
    events["sentence"] = events.groupby("sentence_id").word.transform(
        lambda words: make_sentence(words)[0]
    )
    events["sentence_char"] = events.groupby("sentence_id").word.transform(
        lambda words: make_sentence(words)[1]
    )
    events["context"] = events.groupby("sentence_id").word.transform(
        lambda words: make_sentence(words)[2]
    )
    events = events.rename(columns={"word": "text"})
    events["timeline"] = "svo_word_level"
    events["language"] = "en"
    events["type"] = "Word"
    events["start"] = events.index
    events["duration"] = 0.5
    events = ns.segments.validate_events(events)

    return events

In [ ]:
events = events_from_words(words)
events

In [ ]:
feature = HuggingFaceText(
    contextualized=True,
    token_aggregation="mean",
    model_name="gpt2",
    device="cuda",
    batch_size=64,
    layers=2 / 3,
    cache_all_layers=True,
)

In [ ]:
events = feature._events_from_dataframe(events)

In [ ]:
data = feature._get_data([row for _, row in events.iterrows()])
data = list(data)
data = np.array(data)
data = torch.Tensor(data)
data.shape

In [ ]:
data = list(
    feature._get_timed_arrays(
        events,
        start=0,
        duration=150,
    )
)

In [ ]:
data[0].data